# ScholarTiny — Colab GPU pilot

Notebook này mount Google Drive, nhận source archive từ máy local, chạy pretrain + SFT pilot bằng GPU và lưu artifacts vào Drive.

Pilot dùng dữ liệu synthetic/offline trong repo để không tự chấp thuận điều khoản dataset bên thứ ba.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime
STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ROOT = Path('/content/drive/MyDrive/scholartiny_colab') / STAMP
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Artifacts:', RUN_ROOT)

In [ ]:
from google.colab import files
import shutil, zipfile
uploaded = files.upload()
archive_name = next(iter(uploaded))
source_root = Path('/content/scholartiny')
if source_root.exists():
    shutil.rmtree(source_root)
with zipfile.ZipFile('/content/' + archive_name) as zf:
    zf.extractall('/content')
print('Source ready:', source_root)

In [ ]:
%cd /content/scholartiny
import subprocess, sys
def pip_install(*args):
    cmd = [sys.executable, '-m', 'pip', 'install', *args]
    result = subprocess.run(cmd)
    print('OK' if result.returncode == 0 else 'SKIPPED/FAILED', ' '.join(cmd))
pip_install('-q', '-e', '.[data]')
pip_install('-q', 'packaging', 'ninja', 'einops')
# Optional CUDA backend; the offline pilot below uses the repo reference backend.
pip_install('-q', 'causal-conv1d>=1.4,<2', '--no-build-isolation')
pip_install('-q', 'mamba-ssm>=2.2,<3', '--no-build-isolation')
import torch
print({'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
assert torch.cuda.is_available(), 'Colab chưa bật GPU: Runtime → Change runtime type → T4 GPU'

In [ ]:
import json
from dataset.tokenizer import ByteTokenizer
from dataset.normalize import convert
from dataset.protocol import canonical_json
from dataset.prepare import prepare
from scripts.generate_academic import tool_dialogue
from agent.tools import SCHEMAS
from trainer.train import parser, run

data_root = RUN_ROOT / 'data'
tok_path = data_root / 'tokenizer.json'
data_root.mkdir(parents=True, exist_ok=True)
ByteTokenizer().save(tok_path)

docs = []
for i in range(160):
    text = (f'Synthetic academic record {i}: sample size {i + 10}. '
            f'Recorded value {i + 1}. This is an original fixture for pipeline testing. '
            'An equation relates quantities and a citation identifies the supporting passage.')
    docs.append(dict(convert({'text': text}, {'adapter': 'text', 'min_chars': 1}), split='train',
                     source='colab_synthetic_fixture', provenance={'synthetic': True}))
pretrain_jsonl = data_root / 'pretrain.jsonl'
pretrain_jsonl.write_text(''.join(canonical_json(x) + '\n' for x in docs), encoding='utf-8')

conversations = []
for i in range(64):
    messages = [{'role': 'system', 'content': 'Use tools for exact calculations. Do not invent observations.'}]
    messages += tool_dialogue(f'Calculate {i + 2}+3.', 'math.calculate', {'expression': f'{i + 2}+3'}, str(i + 5))
    conversations.append({'kind': 'sft', 'messages': messages, 'split': 'train', 'tools': [SCHEMAS[0]]})
sft_jsonl = data_root / 'sft.jsonl'
sft_jsonl.write_text(''.join(canonical_json(x) + '\n' for x in conversations), encoding='utf-8')

packed_pretrain = data_root / 'packed_pretrain'
packed_sft = data_root / 'packed_sft'
prepare(pretrain_jsonl, tok_path, packed_pretrain, 'pretrain', 64)
prepare(sft_jsonl, tok_path, packed_sft, 'sft', 256)

cfg = source_root / 'configs' / 'debug.yaml'
common = ['--config', str(cfg), '--tokenizer', str(tok_path), '--device', 'cuda', '--backend', 'reference',
          '--precision', 'fp32', '--max-steps', '20', '--warmup-steps', '4', '--save-every', '5',
          '--batch-size', '1', '--grad-accum', '1']
pre_out = RUN_ROOT / 'checkpoints' / 'pretrain'
pre_ckpt = run(parser().parse_args(common + ['--data', str(packed_pretrain), '--out', str(pre_out), '--seq-len', '64']))
sft_out = RUN_ROOT / 'checkpoints' / 'sft'
sft_ckpt = run(parser().parse_args(common + ['--data', str(packed_sft), '--out', str(sft_out), '--seq-len', '256', '--init', str(pre_ckpt)]))
print('pretrain checkpoint:', pre_ckpt)
print('sft checkpoint:', sft_ckpt)

In [ ]:
import os
for path in sorted(RUN_ROOT.rglob('*')):
    if path.is_file():
        print(f'{path.relative_to(RUN_ROOT)}\t{path.stat().st_size / 1024**2:.2f} MiB')
print('Hoàn tất: data + checkpoints đã nằm trên Google Drive.')